# Continuum Memory System (CMS)

The Continuum Memory System provides **unbounded context** through multi-level memory banks.

## Key Concepts

1. **Multi-level Memory**: Short-term, medium-term, and long-term banks
2. **Frequency-based Storage**: Different levels store at different frequencies
3. **Associative Retrieval**: Content-based memory retrieval
4. **Automatic Consolidation**: Information flows from short-term → long-term

## Architecture

```
Input → Query CMS → Retrieve Relevant Memories → Process → Store to CMS
         ↓                                                       ↓
    [Short-term] ← ← ← ← ← ← ← ← ← ← ← ← ← ← ← ← ← ← ← [Consolidation]
    [Medium-term]
    [Long-term  ]
```

In [ ]:
# Setup
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.memory import ContinuumMemorySystem, AssociativeMemory
from src.layers import CMSBlock
from src.training.metrics import MemoryTracker

torch.manual_seed(42)
np.random.seed(42)

print("✓ Imports successful!")

## Part 1: Basic CMS Operations

In [ ]:
# Create a 3-level CMS
cms = ContinuumMemorySystem(
    memory_config={
        0: {'capacity': 100},   # Short-term: 100 items
        1: {'capacity': 50},    # Medium-term: 50 items
        2: {'capacity': 25}     # Long-term: 25 items
    },
    key_dim=64,
    value_dim=64
)

print("CMS Configuration:")
print(f"  Levels: {cms.num_levels}")
print(f"  Key dim: {cms.key_dim}")
print(f"  Value dim: {cms.value_dim}")
print("\nMemory capacities:")
for level, bank in cms.memory_banks.items():
    print(f"  Level {level}: {bank.capacity} items")

### Store and Retrieve

In [ ]:
# Store some memories
batch_size = 10
seq_len = 5

for step in range(20):
    # Generate random keys and values
    keys = torch.randn(batch_size, seq_len, 64)
    values = torch.randn(batch_size, seq_len, 64)
    
    # Store to CMS
    cms.store(keys, values, step=step)

# Check memory stats
stats = cms.get_memory_stats()
print("\nMemory Statistics after 20 steps:")
for level, level_stats in stats.items():
    print(f"  {level}:")
    print(f"    Count: {level_stats['count']}/{level_stats['capacity']}")
    print(f"    Utilization: {level_stats['utilization']:.2%}")

In [ ]:
# Query memories
query_keys = torch.randn(batch_size, seq_len, 64)

retrieved_values, similarities = cms.query(query_keys, k=5)

print("\nRetrieval Results:")
print(f"  Retrieved values shape: {retrieved_values.shape}")
print(f"  Similarities shape: {similarities.shape}")
print(f"  Average similarity: {similarities.mean():.4f}")
print(f"  Max similarity: {similarities.max():.4f}")

## Part 2: Visualizing Memory Dynamics

In [ ]:
# Track memory usage over time
cms2 = ContinuumMemorySystem(
    memory_config={
        0: {'capacity': 50},
        1: {'capacity': 25},
        2: {'capacity': 10}
    },
    key_dim=32,
    value_dim=32
)

tracker = MemoryTracker(num_levels=3)

# Simulate storing over many steps
num_steps = 200
batch_size = 4
seq_len = 3

for step in range(num_steps):
    keys = torch.randn(batch_size, seq_len, 32)
    values = torch.randn(batch_size, seq_len, 32)
    
    cms2.store(keys, values, step=step)
    
    # Track utilization
    stats = cms2.get_memory_stats()
    tracker.record_memory_usage(step, stats)

print(f"✓ Simulated {num_steps} steps")

In [ ]:
# Plot memory utilization
level_stats = tracker.get_level_statistics()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for level in range(3):
    level_name = ['Short-term', 'Medium-term', 'Long-term'][level]
    stats = level_stats[f'level_{level}']
    
    steps = list(stats['utilization'].keys())
    utils = list(stats['utilization'].values())
    
    axes[level].plot(steps, utils, linewidth=2, color=['blue', 'orange', 'red'][level])
    axes[level].axhline(y=1.0, color='k', linestyle='--', alpha=0.3, label='Capacity')
    axes[level].fill_between(steps, 0, utils, alpha=0.3, color=['blue', 'orange', 'red'][level])
    axes[level].set_xlabel('Step')
    axes[level].set_ylabel('Utilization')
    axes[level].set_title(f'{level_name} Memory')
    axes[level].grid(True, alpha=0.3)
    axes[level].set_ylim([0, 1.1])
    axes[level].legend()

plt.tight_layout()
plt.show()

print("\nFinal Utilization:")
for level in range(3):
    level_name = ['Short-term', 'Medium-term', 'Long-term'][level]
    stats = level_stats[f'level_{level}']
    final_util = list(stats['utilization'].values())[-1]
    print(f"  {level_name}: {final_util:.2%}")

## Part 3: CMSBlock in Action

CMSBlock integrates CMS into a neural network layer.

In [ ]:
# Create a CMSBlock
cms_block = CMSBlock(
    hidden_dim=128,
    memory_config={
        0: {'capacity': 100},
        1: {'capacity': 50},
        2: {'capacity': 25}
    },
    use_memory=True,
    dropout=0.1
)

print("CMSBlock Configuration:")
print(f"  Hidden dim: {cms_block.hidden_dim}")
print(f"  Memory enabled: {cms_block.use_memory}")
print(f"  Memory levels: {cms_block.cms.num_levels}")

In [ ]:
# Process a sequence
batch_size = 8
seq_len = 16
hidden_dim = 128

# Input sequence
x = torch.randn(batch_size, seq_len, hidden_dim)

print(f"Input shape: {x.shape}")

# Forward pass
output, memory_info = cms_block(x, step=0, return_memory_info=True)

print(f"\nOutput shape: {output.shape}")
print(f"Memory info keys: {memory_info.keys()}")
print(f"Retrieved memories shape: {memory_info['retrieved'].shape}")

In [ ]:
# Process multiple steps and track memory
cms_block.reset_memory()

outputs = []
memory_stats = []

for step in range(50):
    x = torch.randn(batch_size, seq_len, hidden_dim)
    output, _ = cms_block(x, step=step)
    outputs.append(output)
    
    # Get memory stats
    stats = cms_block.get_memory_stats()
    memory_stats.append(stats)

print(f"✓ Processed 50 steps")
print(f"\nFinal memory status:")
for level, stats in memory_stats[-1].items():
    print(f"  {level}: {stats['count']}/{stats['capacity']} ({stats['utilization']:.1%})")

## Part 4: Memory Consolidation

Observe how information flows from short-term to long-term memory.

In [ ]:
# Create CMS with small capacities to see consolidation
cms3 = ContinuumMemorySystem(
    memory_config={
        0: {'capacity': 10},  # Small short-term
        1: {'capacity': 5},
        2: {'capacity': 3}
    },
    key_dim=16,
    value_dim=16
)

# Track memory counts
counts_over_time = {0: [], 1: [], 2: []}

for step in range(100):
    # Store memories
    keys = torch.randn(2, 3, 16)
    values = torch.randn(2, 3, 16)
    cms3.store(keys, values, step=step)
    
    # Record counts
    stats = cms3.get_memory_stats()
    for level in range(3):
        counts_over_time[level].append(stats[f'level_{level}']['count'])

# Plot consolidation
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for level in range(3):
    label = ['Short-term', 'Medium-term', 'Long-term'][level]
    color = ['blue', 'orange', 'red'][level]
    plt.plot(counts_over_time[level], label=label, linewidth=2, color=color)
    
    # Show capacity line
    capacity = [10, 5, 3][level]
    plt.axhline(y=capacity, color=color, linestyle='--', alpha=0.3)

plt.xlabel('Step')
plt.ylabel('Memory Count')
plt.title('Memory Consolidation Over Time')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Show utilization
capacities = [10, 5, 3]
for level in range(3):
    label = ['Short-term', 'Medium-term', 'Long-term'][level]
    color = ['blue', 'orange', 'red'][level]
    utilization = [c / capacities[level] for c in counts_over_time[level]]
    plt.plot(utilization, label=label, linewidth=2, color=color)

plt.axhline(y=1.0, color='k', linestyle='--', alpha=0.3, label='Full')
plt.xlabel('Step')
plt.ylabel('Utilization')
plt.title('Memory Utilization Over Time')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Retrieval Quality

Analyze the quality of retrieved memories.

In [ ]:
# Create CMS and store structured data
cms4 = ContinuumMemorySystem(
    memory_config={
        0: {'capacity': 100},
        1: {'capacity': 50},
        2: {'capacity': 25}
    },
    key_dim=32,
    value_dim=32
)

# Store patterns with known structure
# Pattern 1: High values in first half
# Pattern 2: High values in second half

pattern1_keys = []
pattern1_values = []
pattern2_keys = []
pattern2_values = []

for i in range(20):
    # Pattern 1
    key1 = torch.randn(1, 1, 32)
    key1[:, :, :16] += 2.0  # High values in first half
    val1 = torch.ones(1, 1, 32) * 1.0
    
    cms4.store(key1, val1, step=i*2)
    pattern1_keys.append(key1)
    pattern1_values.append(val1)
    
    # Pattern 2
    key2 = torch.randn(1, 1, 32)
    key2[:, :, 16:] += 2.0  # High values in second half
    val2 = torch.ones(1, 1, 32) * 2.0
    
    cms4.store(key2, val2, step=i*2+1)
    pattern2_keys.append(key2)
    pattern2_values.append(val2)

print("✓ Stored 40 memories (20 per pattern)")

In [ ]:
# Query with pattern 1
query1 = torch.randn(1, 1, 32)
query1[:, :, :16] += 2.0

retrieved1, sim1 = cms4.query(query1, k=10)

# Query with pattern 2
query2 = torch.randn(1, 1, 32)
query2[:, :, 16:] += 2.0

retrieved2, sim2 = cms4.query(query2, k=10)

print("Retrieval Results:")
print(f"  Pattern 1 query - Avg similarity: {sim1.mean():.4f}")
print(f"  Pattern 2 query - Avg similarity: {sim2.mean():.4f}")
print(f"  Pattern 1 retrieved values mean: {retrieved1.mean():.4f}")
print(f"  Pattern 2 retrieved values mean: {retrieved2.mean():.4f}")
print("\n(Pattern 1 should retrieve ~1.0, Pattern 2 should retrieve ~2.0)")

## Key Takeaways

1. **Multi-level Storage**: CMS maintains memories at different timescales
2. **Automatic Consolidation**: Information naturally flows to longer-term memory
3. **Associative Retrieval**: Content-based lookup retrieves relevant memories
4. **Unbounded Context**: Can store arbitrary amounts of information
5. **Integration**: CMSBlock seamlessly integrates into neural architectures

## Applications

- **Continual Learning**: Prevent catastrophic forgetting
- **Long Context**: Process sequences longer than attention limits
- **Few-Shot Learning**: Store and retrieve task-relevant information
- **Meta-Learning**: Accumulate knowledge across tasks

**Next**: Notebook 04 explores meta-learning with DMGD.